In [6]:
import pandas as pd
import yfinance as yf
import numpy as np


In [7]:
CBA_Stock = pd.read_csv("./data/CBA_target_ready_2015_present.csv", parse_dates=["Date"])

print(CBA_Stock.head())
print(CBA_Stock.tail())
print(CBA_Stock.shape)
print(CBA_Stock.columns)


        Date       Open       High        Low      Close  Adj Close   Volume  \
0 2015-01-02  84.959686  85.277962  84.661308  85.277962  51.295269   949439   
1 2015-01-05  85.238182  85.775269  85.049202  85.486832  51.420902  1351651   
2 2015-01-06  84.641411  85.337639  84.412651  84.840332  51.032032  2477275   
3 2015-01-07  84.850281  85.029312  84.094376  84.651360  50.918358  2127190   
4 2015-01-08  85.079041  85.188446  84.671249  84.929848  51.085873  1997761   

   Target_t1  Target_t7  
0          1          0  
1          0          0  
2          0          0  
3          1          0  
4          1          0  
           Date        Open        High         Low       Close   Adj Close  \
2851 2026-04-09  180.570007  182.529999  180.470001  182.529999  182.529999   
2852 2026-04-10  181.000000  183.539993  181.000000  183.380005  183.380005   
2853 2026-04-13  183.050003  184.940002  182.619995  183.199997  183.199997   
2854 2026-04-14  185.000000  185.589996  181.94

In [8]:
CBA_Stock.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Target_t1,Target_t7
0,2015-01-02,84.959686,85.277962,84.661308,85.277962,51.295269,949439,1,0
1,2015-01-05,85.238182,85.775269,85.049202,85.486832,51.420902,1351651,0,0
2,2015-01-06,84.641411,85.337639,84.412651,84.840332,51.032032,2477275,0,0
3,2015-01-07,84.850281,85.029312,84.094376,84.651360,50.918358,2127190,1,0
4,2015-01-08,85.079041,85.188446,84.671249,84.929848,51.085873,1997761,1,0


In [9]:
CBA_Stock["Return"] = CBA_Stock["Close"].pct_change()
CBA_Stock["LogReturn"] = np.log(CBA_Stock["Close"] / CBA_Stock["Close"].shift(1))

CBA_Stock["Return_lag1"] = CBA_Stock["Return"].shift(1)
CBA_Stock["Return_lag2"] = CBA_Stock["Return"].shift(2)
CBA_Stock["Return_lag3"] = CBA_Stock["Return"].shift(3)
CBA_Stock["Return_lag5"] = CBA_Stock["Return"].shift(5)

CBA_Stock["RollingMean_5"] = CBA_Stock["Return"].rolling(5).mean()
CBA_Stock["RollingStd_5"] = CBA_Stock["Return"].rolling(5).std()

CBA_Stock["RollingMean_10"] = CBA_Stock["Return"].rolling(10).mean()
CBA_Stock["RollingStd_10"] = CBA_Stock["Return"].rolling(10).std()

CBA_Stock["VolumeChange"] = CBA_Stock["Volume"].pct_change()
CBA_Stock["VolumeMA_5"] = CBA_Stock["Volume"].rolling(5).mean()

print(CBA_Stock.shape)
print(CBA_Stock.isna().sum())

(2856, 21)
Date               0
Open               0
High               0
Low                0
Close              0
Adj Close          0
Volume             0
Target_t1          0
Target_t7          0
Return             1
LogReturn          1
Return_lag1        2
Return_lag2        3
Return_lag3        4
Return_lag5        6
RollingMean_5      5
RollingStd_5       5
RollingMean_10    10
RollingStd_10     10
VolumeChange       3
VolumeMA_5         4
dtype: int64


In [10]:
import numpy as np

window = 14

delta = CBA_Stock["Close"].diff()

gain = (delta.where(delta > 0, 0)).rolling(window).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window).mean()

rs = gain / loss
CBA_Stock["RSI_14"] = 100 - (100 / (1 + rs))

In [11]:
ema_12 = CBA_Stock["Close"].ewm(span=12, adjust=False).mean()
ema_26 = CBA_Stock["Close"].ewm(span=26, adjust=False).mean()

CBA_Stock["MACD"] = ema_12 - ema_26
CBA_Stock["MACD_signal"] = CBA_Stock["MACD"].ewm(span=9, adjust=False).mean()

In [12]:
CBA_Stock["BB_Mid"] = CBA_Stock["Close"].rolling(20).mean()
CBA_Stock["BB_Std"] = CBA_Stock["Close"].rolling(20).std()

CBA_Stock["BB_Upper"] = CBA_Stock["BB_Mid"] + 2 * CBA_Stock["BB_Std"]
CBA_Stock["BB_Lower"] = CBA_Stock["BB_Mid"] - 2 * CBA_Stock["BB_Std"]

In [13]:
# Tenkan-sen
CBA_Stock["Tenkan_sen"] = (
    CBA_Stock["High"].rolling(9).max() +
    CBA_Stock["Low"].rolling(9).min()
) / 2

# Kijun-sen
CBA_Stock["Kijun_sen"] = (
    CBA_Stock["High"].rolling(26).max() +
    CBA_Stock["Low"].rolling(26).min()
) / 2

# NO forward shift (critical)
CBA_Stock["Senkou_A"] = (CBA_Stock["Tenkan_sen"] + CBA_Stock["Kijun_sen"]) / 2

CBA_Stock["Senkou_B"] = (
    CBA_Stock["High"].rolling(52).max() +
    CBA_Stock["Low"].rolling(52).min()
) / 2

In [14]:
CBA_Stock = CBA_Stock.dropna().reset_index(drop=True)

print("Final shape:", CBA_Stock.shape)
print("NaNs left:\n", CBA_Stock.isna().sum())

Final shape: (2803, 32)
NaNs left:
 Date              0
Open              0
High              0
Low               0
Close             0
Adj Close         0
Volume            0
Target_t1         0
Target_t7         0
Return            0
LogReturn         0
Return_lag1       0
Return_lag2       0
Return_lag3       0
Return_lag5       0
RollingMean_5     0
RollingStd_5      0
RollingMean_10    0
RollingStd_10     0
VolumeChange      0
VolumeMA_5        0
RSI_14            0
MACD              0
MACD_signal       0
BB_Mid            0
BB_Std            0
BB_Upper          0
BB_Lower          0
Tenkan_sen        0
Kijun_sen         0
Senkou_A          0
Senkou_B          0
dtype: int64


In [15]:
train_df = CBA_Stock[CBA_Stock["Date"] < "2022-01-01"]
val_df   = CBA_Stock[(CBA_Stock["Date"] >= "2022-01-01") & (CBA_Stock["Date"] < "2024-01-01")]
test_df  = CBA_Stock[CBA_Stock["Date"] >= "2024-01-01"]

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (1722, 32)
Val: (503, 32)
Test: (578, 32)


In [16]:
TARGET = "Target_t1"   # we start with t+1 first

FEATURES = [col for col in CBA_Stock.columns if col not in ["Date", "Target_t1", "Target_t7"]]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("Feature count:", len(FEATURES))

Feature count: 29


In [17]:
inf_counts = np.isinf(X_train).sum()

print("Columns with infinity:")
print(inf_counts[inf_counts > 0])


Columns with infinity:
VolumeChange    4
dtype: int64


In [18]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val   = X_val.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

# Check NaNs after replacing inf
print("Train NaNs:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

print("Val NaNs:")
print(X_val.isna().sum()[X_val.isna().sum() > 0])

print("Test NaNs:")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

Train NaNs:
VolumeChange    4
dtype: int64
Val NaNs:
Series([], dtype: int64)
Test NaNs:
Series([], dtype: int64)


In [19]:
train_valid = X_train.notna().all(axis=1)
val_valid   = X_val.notna().all(axis=1)
test_valid  = X_test.notna().all(axis=1)

X_train = X_train[train_valid]
y_train = y_train[train_valid]

X_val = X_val[val_valid]
y_val = y_val[val_valid]

X_test = X_test[test_valid]
y_test = y_test[test_valid]

print("Cleaned shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Cleaned shapes:
Train: (1718, 29) (1718,)
Val: (503, 29) (503,)
Test: (578, 29) (578,)


In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)



,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [22]:
y_val_pred = model.predict(X_val_scaled)
y_test_pred = model.predict(X_test_scaled)

y_val_prob = model.predict_proba(X_val_scaled)[:, 1]
y_test_prob = model.predict_proba(X_test_scaled)[:, 1]

In [23]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred)
val_auc = roc_auc_score(y_val, y_val_prob)

print("Validation Metrics:")
print("Accuracy:", val_accuracy)
print("F1 Score:", val_f1)
print("AUC:", val_auc)

Validation Metrics:
Accuracy: 0.5188866799204771
F1 Score: 0.675603217158177
AUC: 0.5068625586408012


In [24]:
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_prob)

print("\nTest Metrics:")
print("Accuracy:", test_accuracy)
print("F1 Score:", test_f1)
print("AUC:", test_auc)


Test Metrics:
Accuracy: 0.5640138408304498
F1 Score: 0.7129840546697038
AUC: 0.5091316690424845


In [25]:
cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[ 13 238]
 [ 14 313]]


testing it for t+7

In [26]:
TARGET = "Target_t7"   # we start with t+1 first

FEATURES = [col for col in CBA_Stock.columns if col not in ["Date", "Target_t1", "Target_t7"]]

X_train2 = train_df[FEATURES]
y_train2 = train_df[TARGET]

X_val2 = val_df[FEATURES]
y_val2 = val_df[TARGET]

X_test2 = test_df[FEATURES]
y_test2 = test_df[TARGET]

print("Feature count:", len(FEATURES))

Feature count: 29


In [27]:
inf_counts = np.isinf(X_train2).sum()

print("Columns with infinity:")
print(inf_counts[inf_counts > 0])


Columns with infinity:
VolumeChange    4
dtype: int64


In [28]:
X_train2 = X_train2.replace([np.inf, -np.inf], np.nan)
X_val2   = X_val2.replace([np.inf, -np.inf], np.nan)
X_test2  = X_test2.replace([np.inf, -np.inf], np.nan)

# Check NaNs after replacing inf
print("Train NaNs:")
print(X_train2.isna().sum()[X_train2.isna().sum() > 0])

print("Val NaNs:")
print(X_val2.isna().sum()[X_val2.isna().sum() > 0])

print("Test NaNs:")
print(X_test2.isna().sum()[X_test2.isna().sum() > 0])

Train NaNs:
VolumeChange    4
dtype: int64
Val NaNs:
Series([], dtype: int64)
Test NaNs:
Series([], dtype: int64)


In [29]:
train_valid2 = X_train2.notna().all(axis=1)
val_valid2   = X_val2.notna().all(axis=1)
test_valid2  = X_test2.notna().all(axis=1)

X_train2 = X_train2[train_valid2]
y_train2 = y_train2[train_valid2]

X_val2 = X_val2[val_valid2]
y_val2 = y_val2[val_valid2]

X_test2 = X_test2[test_valid2]
y_test2 = y_test2[test_valid2]

print("Cleaned shapes:")
print("Train:", X_train2.shape, y_train2.shape)
print("Val:", X_val2.shape, y_val2.shape)
print("Test:", X_test2.shape, y_test2.shape)

Cleaned shapes:
Train: (1718, 29) (1718,)
Val: (503, 29) (503,)
Test: (578, 29) (578,)


In [30]:
X_train_scaled2 = scaler.fit_transform(X_train2)
X_val_scaled2   = scaler.transform(X_val2)
X_test_scaled2  = scaler.transform(X_test2)

In [31]:
model.fit(X_train_scaled2, y_train2)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [32]:
y_val_pred2 = model.predict(X_val_scaled2)
y_test_pred2 = model.predict(X_test_scaled2)

y_val_prob2 = model.predict_proba(X_val_scaled2)[:, 1]
y_test_prob2 = model.predict_proba(X_test_scaled2)[:, 1]

In [33]:
val_accuracy2 = accuracy_score(y_val2, y_val_pred2)
val_f12 = f1_score(y_val2, y_val_pred2)
val_auc2 = roc_auc_score(y_val2, y_val_prob2)

print("Validation Metrics:")
print("Accuracy:", val_accuracy2)
print("F1 Score:", val_f12)
print("AUC:", val_auc2)

Validation Metrics:
Accuracy: 0.5347912524850894
F1 Score: 0.693717277486911
AUC: 0.5285714285714286


In [34]:
test_accuracy2 = accuracy_score(y_test2, y_test_pred2)
test_f12 = f1_score(y_test2, y_test_pred2)
test_auc2 = roc_auc_score(y_test2, y_test_prob2)

print("\nTest Metrics:")
print("Accuracy:", test_accuracy2)
print("F1 Score:", test_f12)
print("AUC:", test_auc2)


Test Metrics:
Accuracy: 0.596885813148789
F1 Score: 0.7459105779716467
AUC: 0.47670585308204266


In [35]:
feature_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": model.coef_[0]
})

feature_importance["Abs"] = feature_importance["Coefficient"].abs()

feature_importance = feature_importance.sort_values("Abs", ascending=False)

print(feature_importance.head(10))

        Feature  Coefficient       Abs
26    Kijun_sen    -1.054971  1.054971
20  MACD_signal    -0.553975  0.553975
3         Close     0.546256  0.546256
4     Adj Close     0.509274  0.509274
27     Senkou_A    -0.501398  0.501398
19         MACD     0.461999  0.461999
0          Open    -0.412679  0.412679
5        Volume    -0.300947  0.300947
17   VolumeMA_5    -0.244926  0.244926
18       RSI_14    -0.237083  0.237083


In [36]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)  # NOTE: use NON-scaled data

print("XGBoost trained.")

XGBoost trained.


/opt/anaconda3/envs/course/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [16:51:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [37]:
y_val_pred = xgb_model.predict(X_val)
y_test_pred = xgb_model.predict(X_test)

y_val_prob = xgb_model.predict_proba(X_val)[:, 1]
y_test_prob = xgb_model.predict_proba(X_test)[:, 1]

In [38]:
print("Validation:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("F1:", f1_score(y_val, y_val_pred))
print("AUC:", roc_auc_score(y_val, y_val_prob))

print("\nTest:")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1:", f1_score(y_test, y_test_pred))
print("AUC:", roc_auc_score(y_test, y_test_prob))

Validation:
Accuracy: 0.5506958250497018
F1: 0.6319218241042345
AUC: 0.5311430201597566

Test:
Accuracy: 0.5173010380622838
F1: 0.5181347150259067
AUC: 0.5360088697198971


In [39]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, make_scorer

tscv = TimeSeriesSplit(n_splits=5)

xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10]
}

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=40,
    scoring="roc_auc",
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best CV AUC:", search.best_score_)
print("Best Params:")
print(search.best_params_)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best CV AUC: 0.5126541846671533
Best Params:
{'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 0.01, 'n_estimators': 100, 'min_child_weight': 7, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 1.0}


In [40]:
best_xgb = search.best_estimator_

y_val_pred = best_xgb.predict(X_val)
y_test_pred = best_xgb.predict(X_test)

y_val_prob = best_xgb.predict_proba(X_val)[:, 1]
y_test_prob = best_xgb.predict_proba(X_test)[:, 1]

In [41]:
print("Validation:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("F1:", f1_score(y_val, y_val_pred))
print("AUC:", roc_auc_score(y_val, y_val_prob))

print("\nTest:")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1:", f1_score(y_test, y_test_pred))
print("AUC:", roc_auc_score(y_test, y_test_prob))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Validation:
Accuracy: 0.5089463220675944
F1: 0.5944170771756979
AUC: 0.5202548497527577

Test:
Accuracy: 0.5415224913494809
F1: 0.6324549237170597
AUC: 0.541065097408531

Test Confusion Matrix:
[[ 85 166]
 [ 99 228]]


In [42]:
import requests
import pandas as pd
from datetime import datetime, timezone

def to_utc_timestamp(date_str):
    """
    Convert YYYY-MM-DD string to UTC Unix timestamp.
    """
    return int(datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())

# Test settings
subreddit = "AusFinance"
query = "CBA"
after = to_utc_timestamp("2023-01-01")
before = to_utc_timestamp("2023-02-01")

url = "https://api.pullpush.io/reddit/search/submission/"

params = {
    "subreddit": subreddit,
    "q": query,
    "after": after,
    "before": before,
    "size": 100
}

response = requests.get(url, params=params, timeout=30)

print("Status Code:", response.status_code)
print("Final URL:", response.url)

data = response.json()
posts = data.get("data", [])

print("Number of posts collected:", len(posts))

df_reddit_test = pd.DataFrame(posts)

df_reddit_test.head()

Status Code: 200
Final URL: https://api.pullpush.io/reddit/search/submission/?subreddit=AusFinance&q=CBA&after=1672531200&before=1675209600&size=100
Number of posts collected: 27


,all_awardings,allow_live_comments,archived,author,author_created_utc,author_flair_background_color,author_flair_css_class,author_flair_richtext,author_flair_template_id,author_flair_text,...,total_awards_received,treatment_tags,upvote_ratio,url,view_count,whitelist_status,wls,url_overridden_by_dest,post_hint,preview
0,[],True,False,lambo100,1.365723e+09,None,None,[],None,None,...,0,[],0.79,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN
1,[],False,False,checkyourvitalsigns,1.568690e+09,None,None,[],None,None,...,0,[],0.33,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN
2,[],False,False,QuickBed4623,1.674639e+09,None,None,[],None,None,...,0,[],1.00,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN
3,[],False,False,Remarkable_State_540,1.636827e+09,None,None,[],None,None,...,0,[],0.60,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN
4,[],False,False,arejay007,1.514954e+09,None,None,[],None,None,...,0,[],0.30,https://www.reddit.com/r/AusFinance/comments/1...,None,all_ads,6,NaN,NaN,NaN


In [43]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time

def to_utc_timestamp(date_str):
    return int(datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())

def fetch_month_data(subreddit, after, before):
    url = "https://api.pullpush.io/reddit/search/submission/"
    
    params = {
        "subreddit": subreddit,
        "after": after,
        "before": before,
        "size": 100
    }
    
    response = requests.get(url, params=params, timeout=30)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        return []
    
    data = response.json()
    return data.get("data", [])

# -------- CONFIG --------
subreddits = ["AusFinance", "ASX_Bets"]
start_year = 2015
end_year = 2024   # start smaller first (we expand later)

all_posts = []

# -------- LOOP --------
for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        
        start_date = f"{year}-{month:02d}-01"
        
        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"
        
        after = to_utc_timestamp(start_date)
        before = to_utc_timestamp(end_date)
        
        print(f"Fetching {start_date} to {end_date}")
        
        for sub in subreddits:
            posts = fetch_month_data(sub, after, before)
            print(f"{sub}: {len(posts)} posts")
            
            all_posts.extend(posts)
            
            time.sleep(1)  # prevent rate issues

# -------- SAVE --------
df_reddit = pd.DataFrame(all_posts)

# Keep only important columns
df_reddit = df_reddit[[
    "created_utc",
    "subreddit",
    "title",
    "selftext",
    "score"
]]

# Save to your data folder
file_path = "data/reddit_raw.csv"
df_reddit.to_csv(file_path, index=False)

print("Saved to:", file_path)
print("Total posts:", len(df_reddit))

Fetching 2015-01-01 to 2015-02-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-02-01 to 2015-03-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-03-01 to 2015-04-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-04-01 to 2015-05-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-05-01 to 2015-06-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-06-01 to 2015-07-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-07-01 to 2015-08-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-08-01 to 2015-09-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-09-01 to 2015-10-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-10-01 to 2015-11-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-11-01 to 2015-12-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2015-12-01 to 2016-01-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2016-01-01 to 2016-02-01
AusFinance: 100 posts
ASX_Bets: 0 posts
Fetching 2016-02-01 to 2016-03-01
AusF